## Portable path configuration
This release copy contains no personal or NAS paths. Set `GAO2026_DATA_ROOT` for deposited data and `GAO2026_RAW_ROOT` only when running raw-image stages (raw revision microscopy is not included).


In [ ]:
from pathlib import Path
import os
_default_repo_root = Path.cwd().resolve()
GAO2026_DATA_ROOT = Path(os.environ.get('GAO2026_DATA_ROOT', _default_repo_root / 'data')).expanduser().resolve()
GAO2026_RAW_ROOT = Path(os.environ.get('GAO2026_RAW_ROOT', GAO2026_DATA_ROOT / 'external_raw')).expanduser().resolve()
GAO2026_OUTPUT_ROOT = Path(os.environ.get('GAO2026_OUTPUT_ROOT', _default_repo_root / 'outputs')).expanduser().resolve()
GAO2026_EXTERNAL_ROOT = Path(os.environ.get('GAO2026_EXTERNAL_ROOT', GAO2026_DATA_ROOT / 'external')).expanduser().resolve()


# Fixed-cell SoRa snapshot analysis: six-pixel revision 20260810

This notebook has not been executed after the 20260810 code revision.

The primary state-classification analysis uses a resolution-adapted `r_mass`
radius of 6 SoRa pixels (0.139 um; 0.279-um diameter). This local intensity
aperture is distinct from the geometric mTetR-MCP distance cutoff, which
remains <=0.39 um.

The revised analysis avoids raw ND2 reprocessing:

- reviewed segmentation remains read-only;
- candidate tables and locus crops are linked read-only from the original
  six-pixel SoRa analysis;
- `random_qc_*` arrays are linked read-only from the 20260809 revision, where
  random controls were sampled from all QC-passing nuclei independently of
  mTetR detection and state;
- all threshold, state, Gao-style, focus-QC, and validation outputs are written
  to a new `revision_20260810` output root.

## Cells to run

In a fresh kernel, run code-cell indices **1, 3, 5, 11, 13, 15, 17, 19, 21,
23, and 25**, in that order. Code-cell indices 7 and 9 (Cellpose smoke test and
all-FOV segmentation) do not need to be rerun. Cell 13 uses `overwrite=False`;
it reuses matching links on restart and refuses to replace a mismatched file or
non-empty cache directory.


In [ ]:
from __future__ import annotations

import hashlib
import importlib
import json
import os
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image as DisplayImage
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
CODE_DIR = NOTEBOOK_DIR.parent / 'scripts'
if not CODE_DIR.is_dir():
    CODE_DIR = NOTEBOOK_DIR / 'scripts'
if not CODE_DIR.is_dir():
    raise FileNotFoundError(f'SoRa scripts directory not found from {NOTEBOOK_DIR}')
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

MODULE_FILES = [
    "sora_snapshot_pipeline.py",
    "sora_gao3d_snapshot_pipeline.py",
    "sora_multidataset_pipeline.py",
    "normal_field8_gao_pipeline.py",
    "normal_field8_sora_aligned_pipeline.py",
    "normal_multidataset_pipeline.py",
    "sora_rep1_rep4_pipeline.py",
]
MODULE_HASHES = {
    name: hashlib.sha256((CODE_DIR / name).read_bytes()).hexdigest()
    for name in MODULE_FILES
    if (CODE_DIR / name).is_file()
}
display(pd.Series(MODULE_HASHES, name="sha256").to_frame())
print({"hostname": platform.node(), "python": sys.executable})

# Reload in dependency order so a restarted cell cannot retain the 17-px
# configuration from an earlier kernel state.
import sora_snapshot_pipeline as snapshot
snapshot = importlib.reload(snapshot)
import sora_gao3d_snapshot_pipeline as gao
gao = importlib.reload(gao)
import sora_multidataset_pipeline as legacy_batch
legacy_batch = importlib.reload(legacy_batch)
import sora_rep1_rep4_pipeline as batch
batch = importlib.reload(batch)

BUNDLES = batch.build_dataset_bundles()
AGGREGATE_CONFIG = batch.aggregate_config()
DATASETS = batch.bundle_table(BUNDLES)

display(DATASETS)
assert len(DATASETS) == 24
assert DATASETS["fov_count"].sum() == 8503
assert "revision_20260810" in str(AGGREGATE_CONFIG.output_root)
assert AGGREGATE_CONFIG.final_r_mass_radius_px == 6
assert np.isclose(AGGREGATE_CONFIG.r_mass_pad_um, 0.139, atol=0.002)
print({"aggregate_output": str(AGGREGATE_CONFIG.output_root)})


## 1. ND2 metadata and replicate/field pairing

Re-read all 24 acquisitions from the ND2 files and verify the FOV count,
Z/C/Y/X dimensions, voxel size, channel order, and exposure times.

Analysis hierarchy:

1. Acquisition QC: inspect each ND2 file independently.
2. Biological replicate: combine paired fields from the same replicate and condition.
3. Condition: display Rep1-Rep4 in parallel.
4. All-data pooled: diagnose the overall threshold distributions.

In [ ]:
PREFLIGHT = batch.preflight_datasets(BUNDLES)
POOLING_MEMBERSHIP = batch.pooling_membership_table(BUNDLES)
display(PREFLIGHT)
display(POOLING_MEMBERSHIP)
assert len(PREFLIGHT) == 24
assert int(PREFLIGHT["fov_count"].sum()) == 8503
assert PREFLIGHT["all_checks_passed"].all()
assert len(POOLING_MEMBERSHIP) == 16
assert POOLING_MEMBERSHIP["pooling_check_passed"].all()
SOX2_SER5PH_REP3_POOL = POOLING_MEMBERSHIP.loc[
    (POOLING_MEMBERSHIP["condition_id"] == "sox2_ser5ph")
    & (
        POOLING_MEMBERSHIP["biological_replicate"]
        == "rep3"
    )
].iloc[0]
assert SOX2_SER5PH_REP3_POOL["source_fields"] == "3+6"
assert {
    "sox2_ser5ph_rep3_chamberc_field3",
    "sox2_ser5ph_rep3_chamberc_field6",
}.issubset(
    set(
        SOX2_SER5PH_REP3_POOL[
            "source_analysis_ids"
        ].split(" + ")
    )
)

## 2. Common fixed-cell and resolution-adapted conditions

The reviewed SoRa segmentation remains unchanged. The resolution-adapted LoG
and Trackpy feature scales are retained. The local `r_mass` intensity aperture
uses a 6-pixel radius (0.139 um; 0.279-um diameter), consistent with the SoRa
sampling and spot scale.

The mTetR-MCP distance cutoff remains <=0.39 um and is used only as an
independent geometric criterion for the Active call. It is not the radius used
to calculate `r_mass`.

The fixed-cell thresholds `r_mTetR=1.035` and `r_MCP=1.05` are retained because
fixation changes mTetR localization and intensity. The Methods should report
this as a fixed-cell calibration rather than direct reuse of live-cell numeric
cutoffs.


In [ ]:
COMMON_PARAMETERS = {
    "segmentation": {
        "downsample": 4,
        "background_sigma_full_px": 30,
        "signal_sigma_full_px": 11,
        "cellpose_model": "nuclei",
        "diameter_4x_px": 110,
        "cellprob_threshold": 0.5,
        "flow_threshold": 0.4,
        "dilation_4x_px": 15,
        "random_control_mask": "pre-dilation selected nuclear mask",
    },
    "spot": {
        "median_kernel_zyx": (1, 3, 3),
        "log_kernel_zyx_px": (0.5, 4.0, 4.0),
        "minimum_distance_zyx_px": (1.0, 4.0, 4.0),
        "threshold": "central-plane nuclear mean / 20",
        "r_mass_radius_px": AGGREGATE_CONFIG.final_r_mass_radius_px,
        "r_mass_radius_um": AGGREGATE_CONFIG.r_mass_pad_um,
        "r_mass_diameter_um": 2 * AGGREGATE_CONFIG.r_mass_pad_um,
        "trackpy_diameter_px": 11,
        "trackpy_separation_px": 12,
        "crop_size_px": 101,
    },
    "state": {
        "r_mTetR": 1.035,
        "r_MCP": 1.05,
        "distance_um": 0.39,
        "threshold_context": "fixed-cell acquisition-specific",
    },
    "random_control": {
        "population": "all QC-passing nuclei with valid crop positions",
        "independent_of_mTetR_detection": True,
        "independent_of_r_mass_radius": True,
    },
}
display(pd.json_normalize(COMMON_PARAMETERS, sep=".").T)
assert AGGREGATE_CONFIG.final_r_mass_radius_px == 6
assert np.isclose(AGGREGATE_CONFIG.r_mass_pad_um, 0.139, atol=0.002)
assert np.isclose(AGGREGATE_CONFIG.mcp_near_distance_um, 0.39)


## 3. Cellpose smoke test for newly processed data

Process FOV 0 of the first previously unprocessed dataset (ChamberC Field1)
on an RTX 3090 and inspect preprocessing, the nuclei model, the final dilated
mask, and the overlay. Reuse this cache for full-FOV processing.

In [ ]:
SMOKE_BUNDLE = batch.first_pending_bundle(BUNDLES)
SMOKE = batch.run_segmentation_smoke_test(SMOKE_BUNDLE, fov_id=0)
display(pd.Series(SMOKE, name="value").to_frame())
display(DisplayImage(filename=SMOKE["overlay"]))

## 4. Nuclear segmentation of 24 acquisitions

Reuse the eight existing Rep1/Rep2 acquisitions in read-only mode. Process the
16 Rep3/Rep4 acquisitions with the same Cellpose settings. Because each
acquisition has an independent output directory, use eight acquisition-level
workers (spawn) to reduce the CPU preprocessing and PNG-writing bottleneck.
Scientific settings and FOV order remain unchanged. Include Field 8 in this
stage and inspect the overlays for acquisition quality and selected-nucleus counts.

In [ ]:
SEGMENTATION_SUMMARY_PATH = (
    AGGREGATE_CONFIG.output_root / "tables" / "segmentation_batch_summary.csv"
)
if SEGMENTATION_SUMMARY_PATH.is_file():
    SEGMENTATION_SUMMARY = pd.read_csv(SEGMENTATION_SUMMARY_PATH)
else:
    SEGMENTATION_SUMMARY = batch.run_all_segmentations(
        BUNDLES,
        max_workers=8,
    )
display(SEGMENTATION_SUMMARY)


### Segmentation overlays for 24 acquisitions

Create a montage of FOV 0 from each ND2 file. Retain high-Z and focus-out notes in the panel titles.


In [ ]:
SEGMENTATION_CONTACT = (
    AGGREGATE_CONFIG.output_root
    / "figures"
    / "segmentation_representatives_24_acquisitions.png"
)
if not SEGMENTATION_CONTACT.is_file():
    SEGMENTATION_CONTACT = batch.representative_segmentation_contact(BUNDLES)
display(DisplayImage(filename=str(SEGMENTATION_CONTACT)))


## 5. Read-only six-pixel locus and all-QC random-control caches

Raw ND2 quantification is not repeated. Cell 13 creates a new, date-stamped
analysis root whose `per_fov_cache` links to the original 6-pixel candidate and
locus-crop source. Its separate `random_control_cache` links to the 20260809
all-QC random-control source. The `random_qc_*` arrays contain image crops and
sampling coordinates only and do not depend on the `r_mass` aperture.

The setup is restartable and uses `overwrite=False`. Existing 6-pixel and
17-pixel source outputs are never modified.


In [ ]:
CACHE_SOURCE_SUMMARY = batch.seed_revised_six_pixel_caches(
    BUNDLES,
    overwrite=True,
)
display(CACHE_SOURCE_SUMMARY)
assert len(CACHE_SOURCE_SUMMARY) == 24
assert int(CACHE_SOURCE_SUMMARY["fov_count"].sum()) == 8503
assert (
    CACHE_SOURCE_SUMMARY["n_all_qc_random_controls"].sum()
    > 0
)

QUANTIFICATION_SUMMARY_PATH = (
    AGGREGATE_CONFIG.output_root / "tables" / "quantification_batch_summary.csv"
)
if QUANTIFICATION_SUMMARY_PATH.is_file():
    QUANTIFICATION_SUMMARY = pd.read_csv(QUANTIFICATION_SUMMARY_PATH)
else:
    QUANTIFICATION_SUMMARY = batch.run_all_quantifications(
        BUNDLES,
        allow_raw_recompute=False,
    )
display(QUANTIFICATION_SUMMARY)
assert QUANTIFICATION_SUMMARY["method"].eq(
    "read_only_composed_six_pixel_locus_and_all_qc_random"
).all()


### Primary 3D locus overlays for 24 acquisitions


In [ ]:
SPOT_CONTACT = (
    AGGREGATE_CONFIG.output_root
    / "figures"
    / "spot_detection_representatives_24_acquisitions.png"
)
if not SPOT_CONTACT.is_file():
    SPOT_CONTACT = batch.representative_spot_contact(BUNDLES)
display(DisplayImage(filename=str(SPOT_CONTACT)))


## 6. Fixed-cell threshold diagnostics

Diagnostic distributions are saved at acquisition, biological-replicate,
condition, and pooled levels. State calls remain fixed at
`r_mTetR=1.035`, `r_MCP=1.05`, and MCP distance <=0.39 um. These numerical
values are documented as fixed-cell thresholds because fixation changes mTetR
localization and intensity.


In [ ]:
THRESHOLD_TABLES = AGGREGATE_CONFIG.output_root / "tables"
THRESHOLD_FIGURES = AGGREGATE_CONFIG.output_root / "figures"
THRESHOLD_REQUIRED = [
    THRESHOLD_TABLES / "threshold_diagnostics_by_acquisition.csv",
    THRESHOLD_TABLES / "threshold_diagnostics_by_biological_replicate.csv",
    THRESHOLD_TABLES / "threshold_diagnostics_by_condition.csv",
    AGGREGATE_CONFIG.output_root / "threshold_decision.json",
]
if all(path.is_file() for path in THRESHOLD_REQUIRED):
    THRESHOLD_RESULTS = {
        "acquisition_diagnostics": pd.read_csv(THRESHOLD_REQUIRED[0]),
        "biological_replicate_diagnostics": pd.read_csv(THRESHOLD_REQUIRED[1]),
        "condition_diagnostics": pd.read_csv(THRESHOLD_REQUIRED[2]),
        "pooled_decision": json.loads(THRESHOLD_REQUIRED[3].read_text()),
        "pooled_figure": THRESHOLD_FIGURES / "threshold_calibration_four_panel.png",
        "acquisition_contact_sheet": THRESHOLD_FIGURES / "threshold_calibration_by_acquisition.png",
        "biological_replicate_contact_sheet": THRESHOLD_FIGURES / "threshold_calibration_by_biological_replicate.png",
        "condition_contact_sheet": THRESHOLD_FIGURES / "threshold_calibration_by_condition.png",
        "pooled_candidate_count": len(
            pd.read_csv(
                THRESHOLD_TABLES / "all_acquisitions_spot_candidates.csv",
                usecols=[0],
            )
        ),
    }
else:
    THRESHOLD_RESULTS = batch.run_threshold_calibrations(BUNDLES)
display(THRESHOLD_RESULTS["acquisition_diagnostics"])
display(THRESHOLD_RESULTS["biological_replicate_diagnostics"])
display(THRESHOLD_RESULTS["condition_diagnostics"])
display(DisplayImage(filename=str(THRESHOLD_RESULTS["acquisition_contact_sheet"])))
display(DisplayImage(filename=str(THRESHOLD_RESULTS["biological_replicate_contact_sheet"])))
display(DisplayImage(filename=str(THRESHOLD_RESULTS["condition_contact_sheet"])))


## 7. Pooled threshold diagnostic across all 24 acquisitions

Because mTetR/MCP exposure, laser power, voxel size, and analysis radius are
matched, combine all candidates for a diagnostic plot while retaining dataset
ID, replicate, and chamber.

The 640-nm mintbody exposure is 100 ms only in Rep2 and 200 ms otherwise, so
do not directly pool absolute 640-nm intensities across replicates. State
thresholding uses only the 515- and 445-nm channels.

In [ ]:
display(
    pd.Series(
        THRESHOLD_RESULTS["pooled_decision"],
        name="value",
    ).to_frame()
)
print(
    "Pooled candidate rows:",
    f"{THRESHOLD_RESULTS['pooled_candidate_count']:,}",
)
display(DisplayImage(filename=str(THRESHOLD_RESULTS["pooled_figure"])))

## 8. State calls and Gao-style locus aggregation

The all-acquisition analysis remains primary. State calls use the 6-pixel
`r_mass` values with fixed thresholds `1.035 / 1.05`; the separate mTetR-MCP
distance criterion remains <=0.39 um. Random-mean subtraction uses all
QC-passing nuclei with valid random crops rather than only nuclei with a
detected rank-1 mTetR locus. Biological samples are formed only after
acquisition-level state calling and QC.


In [ ]:
STATE_TABLES = AGGREGATE_CONFIG.output_root / "tables"
STATE_FIGURES = AGGREGATE_CONFIG.output_root / "figures"
STATE_REQUIRED = [
    STATE_TABLES / "state_summary_by_acquisition.csv",
    STATE_TABLES / "state_summary_by_biological_replicate.csv",
    STATE_TABLES / "state_summary_by_condition.csv",
    STATE_TABLES / "state_summary_all_rep1_rep4_pooled.csv",
    STATE_TABLES / "biological_sample_pooling_membership.csv",
    STATE_FIGURES / "gao_fig1c_by_biological_replicate.png",
]
if all(path.is_file() for path in STATE_REQUIRED):
    biological_summary = pd.read_csv(STATE_REQUIRED[1])
    STATE_RESULTS = {
        "acquisition_summary": pd.read_csv(STATE_REQUIRED[0]),
        "biological_replicate_summary": biological_summary,
        "paired_field_summary": biological_summary.loc[
            biological_summary["biological_replicate"].isin(["rep3", "rep4"])
        ].reset_index(drop=True),
        "pooling_membership": pd.read_csv(STATE_REQUIRED[4]),
        "condition_summary": pd.read_csv(STATE_REQUIRED[2]),
        "pooled_summary": pd.read_csv(STATE_REQUIRED[3]),
        "state_counts_figure": STATE_FIGURES / "state_counts_by_acquisition.png",
        "biological_replicate_figure": STATE_FIGURES / "active_fraction_by_biological_replicate.png",
        "paired_field_figure": STATE_FIGURES / "paired_field_pooled_results_rep3_rep4.png",
        "gao_contact_sheet": STATE_REQUIRED[5],
    }
else:
    STATE_RESULTS = batch.run_states_and_locus_summaries(BUNDLES)
display(STATE_RESULTS["acquisition_summary"])
display(STATE_RESULTS["biological_replicate_summary"])
display(STATE_RESULTS["paired_field_summary"])
display(STATE_RESULTS["pooling_membership"])
display(STATE_RESULTS["condition_summary"])
display(STATE_RESULTS["pooled_summary"])
display(DisplayImage(filename=str(STATE_RESULTS["state_counts_figure"])))
display(DisplayImage(filename=str(STATE_RESULTS["biological_replicate_figure"])))
display(DisplayImage(filename=str(STATE_RESULTS["paired_field_figure"])))
if STATE_RESULTS["gao_contact_sheet"] is not None:
    display(DisplayImage(filename=str(STATE_RESULTS["gao_contact_sheet"])))


## 9. Focus-QC sensitivity analysis

Two acquisition-level QC notes are retained: ChamberC Field 8 was acquired
substantially above the intended plane and ChamberD Field 8 was designated
focus-out. The primary analysis includes all acquisitions. This separate
sensitivity analysis excludes only those two acquisitions and recomputes the
affected Nanog-Ser5ph Rep3/Rep4 locus summaries.


In [ ]:
FOCUS_ROOT = AGGREGATE_CONFIG.output_root / "focus_qc_sensitivity"
FOCUS_REQUIRED = [
    FOCUS_ROOT / "tables" / "excluded_acquisitions.csv",
    FOCUS_ROOT / "tables" / "focus_qc_sensitivity_comparison.csv",
    FOCUS_ROOT / "figures" / "focus_qc_passing_gao_summaries.png",
]
if all(path.is_file() for path in FOCUS_REQUIRED):
    FOCUS_QC = {
        "flagged_acquisitions": pd.read_csv(FOCUS_REQUIRED[0]),
        "comparison": pd.read_csv(FOCUS_REQUIRED[1]),
        "contact_sheet": FOCUS_REQUIRED[2],
    }
else:
    FOCUS_QC = batch.run_focus_qc_sensitivity(BUNDLES)
display(FOCUS_QC["flagged_acquisitions"])
display(FOCUS_QC["comparison"])
display(DisplayImage(filename=str(FOCUS_QC["contact_sheet"])))


## 10. Completeness validation

Validate all 24 acquisition outputs, 8,503 linked six-pixel locus caches,
8,503 linked all-QC random-control caches, fixed-cell state calls,
biological-replicate summaries, and focus-QC sensitivity outputs.


In [ ]:
BATCH_REPORT = batch.finalize_batch(BUNDLES)
display(pd.Series(BATCH_REPORT["validations"], name="passed").to_frame())
print("All validations passed:", BATCH_REPORT["all_validations_passed"])
print(
    "Batch manifest:",
    AGGREGATE_CONFIG.output_root / "batch_analysis_manifest.json",
)

## Interpretation notes

- The all-acquisition result is primary; the focus-QC exclusion is explicitly
  labeled as sensitivity analysis.
- The 6-pixel `r_mass` aperture has a 0.139-um radius (0.279-um diameter) and is
  independent of the <=0.39-um mTetR-MCP distance criterion.
- Fixed-cell thresholds and segmentation need not numerically match live-cell
  analysis because fixation changes mTetR localization and intensity; this is
  documented in the Methods.
- Higher-resolution persistence of Active-associated locus-centered signal
  supports localization within the SoRa optical-resolution volume, but does
  not establish molecular colocalization or a sub-resolution distance.


In [ ]:
# Keep imports, CONDITIONS, REPLICATES, format_pvalue(),
# and load_snaptag_group() unchanged.

FONT_SIZE = 19.5

plt.rcParams.update(
    {
        "font.family": "Arial",
        "font.size": FONT_SIZE,
        "axes.titlesize": FONT_SIZE,
        "axes.labelsize": FONT_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
        "legend.fontsize": FONT_SIZE,
        "axes.linewidth": 0.8,
        "mathtext.fontset": "custom",
        "mathtext.rm": "Arial",
        "mathtext.it": "Arial:italic",
        "mathtext.bf": "Arial:bold",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
)

STATE_COLORS = {
    "Active": "tab:orange",
    "Inactive": "tab:blue",
    "Random": "tab:green",
}

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def add_scale_bar(axis, image_shape, pixel_size_um: float) -> None:
    bar_um = 0.5
    bar_px = bar_um / pixel_size_um
    _, width = image_shape

    x_end = width - 6
    x_start = x_end - bar_px
    y = 7

    axis.plot(
        [x_start, x_end],
        [y, y],
        color="white",
        linewidth=2.2,
        solid_capstyle="butt",
    )
    axis.text(
        (x_start + x_end) / 2,
        y + 3.5,
        "",
        color="white",
        ha="center",
        va="bottom",
        fontsize=FONT_SIZE,
        path_effects=[
            PathEffects.withStroke(
                linewidth=1.6,
                foreground="black",
            )
        ],
    )


def plot_violin(axis, values: pd.DataFrame, statistics: pd.Series) -> None:
    active = values.loc[
        values["state"].eq("Active"),
        "core_minus_annulus",
    ].dropna().to_numpy(float)

    inactive = values.loc[
        values["state"].eq("Inactive"),
        "core_minus_annulus",
    ].dropna().to_numpy(float)

    if not len(active) or not len(inactive):
        raise RuntimeError(
            "Active and Inactive values are both required."
        )

    violins = axis.violinplot(
        [active, inactive],
        positions=[0, 1],
        widths=0.78,
        showmeans=False,
        showmedians=False,
        showextrema=False,
        points=100,
    )

    for body, state in zip(
        violins["bodies"],
        ("Active", "Inactive"),
    ):
        body.set_facecolor(STATE_COLORS[state])
        body.set_edgecolor("black")
        body.set_linewidth(0.7)
        body.set_alpha(0.72)

    medians = [
        float(np.median(active)),
        float(np.median(inactive)),
    ]
    axis.scatter(
        [0, 1],
        medians,
        marker="_",
        s=190,
        linewidths=1.8,
        color="black",
        zorder=3,
    )

    axis.axhline(
        0,
        color="#777777",
        linewidth=0.8,
        linestyle=":",
    )
    axis.set_xticks(
        [0, 1],
        ["Act.", "Inac."],
    )
    axis.set_xlim(-0.55, 1.55)
    axis.margins(y=0.08)

    # Active-Inactive significance bracket
    y_min, y_max = axis.get_ylim()
    y_range = max(y_max - y_min, 1.0)

    bracket_y = y_max + 0.035 * y_range
    bracket_tick = 0.025 * y_range
    text_y = bracket_y + 0.025 * y_range

    axis.plot(
        [0, 0, 1, 1],
        [
            bracket_y - bracket_tick,
            bracket_y,
            bracket_y,
            bracket_y - bracket_tick,
        ],
        color="black",
        linewidth=0.9,
        clip_on=False,
    )

    axis.text(
        0.5,
        text_y,
        format_pvalue(
            float(statistics["pvalue_two_sided"])
        ),
        ha="center",
        va="bottom",
        fontsize=FONT_SIZE,
    )

    # Extend the y-axis so that the bracket and p value are not clipped.
    axis.set_ylim(
        y_min,
        text_y + 0.055 * y_range,
    )

    axis.grid(
        axis="y",
        color="#D0D0D0",
        linestyle=":",
        linewidth=0.7,
    )
    axis.spines[["top", "right"]].set_visible(False)


group_data = {
    (condition_id, replicate): load_snaptag_group(
        condition_id,
        replicate,
    )
    for condition_id, _ in CONDITIONS
    for replicate in REPLICATES
}

figure = plt.figure(
    figsize=(22.0, 13.0),
    constrained_layout=False,
)

grid = figure.add_gridspec(
    nrows=len(CONDITIONS),
    ncols=13,
    width_ratios=[
        1.0,
        1.0,
        0.20,
        1.30,
        0.16,
        0.95,
        0.38,
        1.0,
        1.0,
        0.20,
        1.30,
        0.16,
        0.95,
    ],
    hspace=0.45,
    wspace=0.12,
)

pixel_size_um = float(
    AGGREGATE_CONFIG.expected_xy_um
)

for row_index, (condition_id, row_label) in enumerate(
    CONDITIONS
):
    for replicate_index, replicate in enumerate(
        REPLICATES
    ):
        data = group_data[
            (condition_id, replicate)
        ]

        # The two replicates are shown side by side without
        # displaying "REP3" or "REP4".
        column_offset = (
            0 if replicate_index == 0 else 7
        )

        active_axis = figure.add_subplot(
            grid[row_index, column_offset]
        )
        inactive_axis = figure.add_subplot(
            grid[row_index, column_offset + 1]
        )
        radial_axis = figure.add_subplot(
            grid[row_index, column_offset + 3]
        )
        violin_axis = figure.add_subplot(
            grid[row_index, column_offset + 5]
        )

        active_image = data["active_image"]
        inactive_image = data["inactive_image"]

        vabs = max(
            float(np.nanmax(np.abs(active_image))),
            float(np.nanmax(np.abs(inactive_image))),
            1.0,
        )
        image_norm = TwoSlopeNorm(
            vmin=-vabs,
            vcenter=0.0,
            vmax=vabs,
        )

        active_axis.imshow(
            active_image,
            cmap="coolwarm",
            norm=image_norm,
            origin="lower",
            interpolation="nearest",
        )
        inactive_axis.imshow(
            inactive_image,
            cmap="coolwarm",
            norm=image_norm,
            origin="lower",
            interpolation="nearest",
        )

        values = data["values"]
        n_active = int(
            values["state"].eq("Active").sum()
        )
        n_inactive = int(
            values["state"].eq("Inactive").sum()
        )

        for axis, sample_size in (
            (active_axis, n_active),
            (inactive_axis, n_inactive),
        ):
            axis.set_xticks([])
            axis.set_yticks([])
            axis.set_aspect("equal")

            axis.text(
                0.03,
                0.03,
                rf"$\it{{N}}$ = {sample_size:,}",
                transform=axis.transAxes,
                color="white",
                ha="left",
                va="bottom",
                fontsize=FONT_SIZE,
                path_effects=[
                    PathEffects.withStroke(
                        linewidth=1.8,
                        foreground="black",
                    )
                ],
            )

            for spine in axis.spines.values():
                spine.set_linewidth(0.7)
                spine.set_color("#555555")

        add_scale_bar(
            inactive_axis,
            inactive_image.shape,
            pixel_size_um,
        )

        radial = data["radial"]

        for state in (
            "Active",
            "Inactive",
            "Random",
        ):
            profile = radial.loc[
                radial["state"].eq(state)
            ].sort_values("radius_um")

            if profile.empty:
                continue

            radius = profile[
                "radius_um"
            ].to_numpy(float)
            mean = profile[
                "mean_scaled_intensity"
            ].to_numpy(float)
            sem = profile[
                "sem_scaled_intensity"
            ].to_numpy(float)

            radial_axis.plot(
                radius,
                mean,
                color=STATE_COLORS[state],
                linewidth=1.8,
                label=state,
            )
            radial_axis.fill_between(
                radius,
                mean - sem,
                mean + sem,
                color=STATE_COLORS[state],
                alpha=0.20,
                linewidth=0,
            )

        radial_axis.axhline(
            0,
            color="#777777",
            linewidth=0.8,
            linestyle=":",
        )
        radial_axis.set_xlabel("Radius (µm)")
        radial_axis.grid(
            color="#D0D0D0",
            linestyle=":",
            linewidth=0.7,
        )
        radial_axis.spines[
            ["top", "right"]
        ].set_visible(False)

        plot_violin(
            violin_axis,
            data["values"],
            data["statistics"],
        )

        if row_index == 0:
            active_axis.set_title(
                "Active",
                pad=10,
            )
            inactive_axis.set_title(
                "Inactive",
                pad=10,
            )
            radial_axis.set_title(
                "Radial intensity\n(max = 100)",
                pad=10,
            )
            violin_axis.set_title(
                "Central enrichment\n",
                pad=10,
            )

        if replicate_index == 0:
            active_axis.text(
                -0.34,
                0.5,
                row_label,
                transform=active_axis.transAxes,
                ha="right",
                va="center",
                fontsize=FONT_SIZE,
            )

figure.legend(
    handles=[
        Line2D(
            [0],
            [0],
            color=STATE_COLORS[state],
            linewidth=2.0,
            label=state,
        )
        for state in (
            "Active",
            "Inactive",
            "Random",
        )
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, 0.992),
    ncol=3,
    frameon=False,
    fontsize=FONT_SIZE,
)

figure.subplots_adjust(
    left=0.10,
    right=0.995,
    top=0.91,
    bottom=0.06,
)

output_pdf = (
    OUTPUT_DIR
    / "SoRa_rep3_rep4_SNAPtag_composite_revised.pdf"
)

figure.savefig(
    output_pdf,
    dpi=300,
    bbox_inches="tight",
)

print(f"Saved -> {output_pdf}")
plt.show()